# Audio Editing with DDPM Inversion

**Zero-Shot Unsupervised and Text-Based Audio Editing Using DDPM Inversion** (ICML 2024)

This notebook demonstrates how to edit audio using the methods from the paper:
1. **Text-Based Editing** - Edit an audio clip by providing a text description of the desired output
2. **Unsupervised Editing** - Edit audio without any text prompts by extracting and applying principal components
3. **Music Remixing** - Remix audio using style transfer, latent blending, or temporal mixing

Supported input formats: **WAV, MP3, FLAC, OGG**
Supported models: AudioLDM, AudioLDM2, TANGO, and Stable Audio Open.

## 1. Setup

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/BF667/AudioEditing.git
%cd AudioEditing
%pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, 'code')

import os
import torch
import torchaudio
from IPython.display import Audio, display
from models import load_model
from utils import load_audio, get_spec, set_reproducability
from torch import inference_mode

## 2. Load Model

Choose one of the supported models. `cvssp/audioldm2-music` is the default and works well for music editing.

In [ ]:
# @markdown ### Model Selection
MODEL_ID = "cvssp/audioldm2-music"  # @param ["cvssp/audioldm2-music", "cvssp/audioldm2", "cvssp/audioldm2-large", "cvssp/audioldm-s-full-v2", "cvssp/audioldm-l-full", "declare-lab/tango-full-ft-audio-music-caps", "declare-lab/tango-full-ft-audiocaps"]
NUM_STEPS = 200  # @param {type:"integer"}
# Use 200 steps for AudioLDM2/TANGO, 100 steps for AudioLDM

device = "cuda:0"
torch.cuda.set_device(0)

print(f"Loading {MODEL_ID}...")
ldm_stable = load_model(MODEL_ID, device, NUM_STEPS)
print("Model loaded!")

## 3. Text-Based Audio Editing

Provide an input audio file and a text prompt describing the desired edit. The model will invert the audio into the diffusion latent space, then generate a new version guided by your prompt.

In [ ]:
# @markdown ### Upload your audio file (WAV, MP3, FLAC, or OGG)
from google.colab import files

print("Upload an audio file (WAV, MP3, FLAC, or OGG):")
uploaded = files.upload()
AUDIO_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {AUDIO_PATH}")

# Listen to the original
wav, sr = torchaudio.load(AUDIO_PATH)
display(Audio(wav.squeeze().numpy(), rate=sr))

In [ ]:
# @markdown ### Editing Parameters
SOURCE_PROMPT = ""  # @param {type:"string"}
TARGET_PROMPT = "a jazz version of this song"  # @param {type:"string"}
CFG_SRC = 3.0  # @param {type:"number"}
CFG_TAR = 12.0  # @param {type:"number"}
TSTART = 100  # @param {type:"integer"}
# Higher tstart = stronger edit. Range: 40-200.
# tstart=100 is the default used in the paper's user study.

from ddm_inversion.inversion_utils import inversion_forward_process, inversion_reverse_process

set_reproducability(seed=42, extreme=False)

# Load audio into model format
x0, sr, duration = load_audio(AUDIO_PATH, ldm_stable.get_fn_STFT(), device=device,
                              stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())
torch.cuda.empty_cache()

# VAE encode
with inference_mode():
    w0 = ldm_stable.vae_encode(x0)

# Forward (inversion) process
print("Running inversion (forward process)...")
with inference_mode():
    wt, zs, wts, extra_info = inversion_forward_process(
        ldm_stable, w0, etas=1.0,
        prompts=[SOURCE_PROMPT], cfg_scales=[CFG_SRC],
        prog_bar=True,
        num_inference_steps=NUM_STEPS,
        numerical_fix=True,
        duration=duration)

# Reverse (generation) process with target prompt
tstart_tensor = torch.tensor([TSTART], dtype=torch.int)
skip = NUM_STEPS - tstart_tensor

print("Generating edited audio (reverse process)...")
with inference_mode():
    w0_edited, _ = inversion_reverse_process(
        ldm_stable, xT=wts, tstart=tstart_tensor,
        fix_alpha=0.1, etas=1.0,
        prompts=[TARGET_PROMPT], neg_prompts=[""],
        cfg_scales=[CFG_TAR], prog_bar=True,
        zs=zs[:int(NUM_STEPS - min(skip))],
        duration=duration, extra_info=extra_info)

# VAE decode
print("Decoding...")
with inference_mode():
    x0_dec = ldm_stable.vae_decode(w0_edited)
    if x0_dec.dim() < 4:
        x0_dec = x0_dec[None, :, :, :]
    with torch.no_grad():
        audio_edited = ldm_stable.decode_to_mel(x0_dec)
        audio_orig = ldm_stable.decode_to_mel(x0)

# Save results (output format matches input: MP3 in -> MP3 out)
os.makedirs("results", exist_ok=True)
in_ext = os.path.splitext(AUDIO_PATH)[1].lstrip('.') or 'wav'
out_path = f"results/edited.{in_ext}"
out_path_orig = f"results/original.{in_ext}"
torchaudio.save(out_path, audio_edited, sample_rate=sr, format=in_ext if in_ext != 'wav' else None)
torchaudio.save(out_path_orig, audio_orig, sample_rate=sr, format=in_ext if in_ext != 'wav' else None)
print(f"Results saved to {out_path}")

In [ ]:
# @markdown ### Listen to results
print("Original audio:")
display(Audio(audio_orig.squeeze().cpu().numpy(), rate=sr))

print(f"Edited audio (prompt: \"{TARGET_PROMPT}\"):")
display(Audio(audio_edited.squeeze().cpu().numpy(), rate=sr))

## 4. Unsupervised Editing (Principal Components)

This method edits audio without text prompts by extracting principal components (PCs) from the diffusion trajectory and applying them with controllable strength.

In [ ]:
# @markdown ### Upload audio for unsupervised editing (or reuse from above)
from google.colab import files

print("Upload an audio file (WAV, MP3, FLAC, or OGG), or skip to use the previous one:")
try:
    uploaded = files.upload()
    UNSUP_AUDIO_PATH = list(uploaded.keys())[0]
except:
    UNSUP_AUDIO_PATH = AUDIO_PATH
    print(f"Using previous file: {UNSUP_AUDIO_PATH}")

In [ ]:
# @markdown ### Step 1: Extract Principal Components
SOURCE_PROMPT_UNSUP = ""  # @param {type:"string"}
DRIFT_START = 200  # @param {type:"integer"}
DRIFT_END = 100  # @param {type:"integer"}
N_EVS = 3  # @param {type:"integer"}
# Number of eigenvectors to extract. More = more editing directions.

from pc_drift import forward_directional, PCStreamChoice, get_eigenvectors
from utils import get_text_embeddings
import numpy as np
from tqdm import tqdm

set_reproducability(seed=42)

x0_u, sr_u, duration_u = load_audio(UNSUP_AUDIO_PATH, ldm_stable.get_fn_STFT(), device=device,
                                     stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())

with inference_mode():
    w0_u = ldm_stable.vae_encode(x0_u)

torch.cuda.empty_cache()
text_emb_u, uncond_emb_u = get_text_embeddings([SOURCE_PROMPT_UNSUP], [""], ldm_stable)[1:]

# Forward inversion
from ddm_inversion.inversion_utils import inversion_forward_process

with inference_mode():
    _, zs_u, wts_u, extra_info_u = inversion_forward_process(
        ldm_stable, w0_u, etas=1.0,
        prompts=[SOURCE_PROMPT_UNSUP], cfg_scales=[3],
        prog_bar=True, num_inference_steps=NUM_STEPS,
        numerical_fix=True, duration=duration_u)

wts_u = wts_u.flip(0)
latents_u = [wts_u[0].unsqueeze(0), *[z.unsqueeze(0) for z in zs_u.flip(0)]]
del wts_u, zs_u
torch.cuda.empty_cache()

timesteps = ldm_stable.model.scheduler.timesteps
drift_start_it = NUM_STEPS - DRIFT_START
drift_end_it = NUM_STEPS - DRIFT_END

mask = torch.ones_like(latents_u[0])

# Extract PCs
print("Extracting principal components...")
xt_u = latents_u[0]
eigdata = {}
prev_pc = None

for it, t in tqdm(enumerate(timesteps), total=len(timesteps)):
    xt_u, x0_pred = forward_directional(ldm_stable, xt_u, t, latents_u[it+1],
                                         uncond_emb_u, text_emb_u, 3, eta=1.0)

    if it >= drift_start_it and it < drift_end_it:
        eigvecs, eigval, _, _, _, _ = get_eigenvectors(
            ldm_stable, xt_u, text_emb_u, uncond_emb_u, latents_u[it+1],
            mask, t, x0_pred, PCStreamChoice.BOTH, 1e-3, 3, 50, False, 1.0, N_EVS)

        if prev_pc is not None:
            corr = (prev_pc.reshape(N_EVS, -1) @ eigvecs.reshape(N_EVS, -1).T).diag()
            for ev_num in range(N_EVS):
                if corr[ev_num] <= -0.8:
                    eigvecs[ev_num] *= -1

        prev_pc = eigvecs
        eigdata[t.item()] = {
            'eigvec': eigvecs.detach().cpu(),
            'eigval': eigval.detach().cpu(),
            'norm_factor': torch.sqrt(ldm_stable.model.scheduler.alphas_cumprod[t])}

    xt_u = xt_u

print(f"Extracted PCs from timesteps {DRIFT_START} to {DRIFT_END}")

In [ ]:
# @markdown ### Step 2: Apply PCs with controllable strength
EVS = [1]  # @param {type:"raw"}
AMOUNT = 1.0  # @param {type:"number"}
# Amount controls editing strength. Positive or negative values work.
# Try values like -2, -1, 0.5, 1, 2 for different effects.

from pc_drift import forward_directional, PCStreamChoice, apply_drift
from utils import get_text_embeddings
import gc

# Reload and setup
text_emb_a, uncond_emb_a = get_text_embeddings([SOURCE_PROMPT_UNSUP], [""], ldm_stable)[1:]
ldm_stable.setup_extra_inputs(latents_u[0], extra_info=extra_info_u,
                              init_timestep=timesteps[0], audio_end_in_s=duration_u)

xt_a = latents_u[0].unsqueeze(0)
drift_start_it_a = NUM_STEPS - DRIFT_START
drift_end_it_a = NUM_STEPS - DRIFT_END

print("Applying principal components...")
for it, t in tqdm(enumerate(timesteps), total=len(timesteps)):
    lat = latents_u[it+1]
    if lat.dim() == 3:
        lat = lat.unsqueeze(0)

    xt_m1, x0_pred = forward_directional(ldm_stable, xt_a, t, lat,
                                         uncond_emb_a, text_emb_a, 3, eta=1.0)

    if it >= drift_start_it_a and it < drift_end_it_a:
        xt_m1 = apply_drift(ldm_stable, xt_m1, x0_pred, t, timesteps,
                           NUM_STEPS, eigdata, lat, device,
                           use_shifted_x0_for_noisepred=True,
                           amount=AMOUNT, ev_nums=EVS, eta=1.0)
    del x0_pred
    xt_a = xt_m1

gc.collect()
torch.cuda.empty_cache()

# Decode
with inference_mode():
    x0_dec_u = ldm_stable.vae_decode(xt_a)
    if x0_dec_u.dim() < 4:
        x0_dec_u = x0_dec_u[None, :, :, :]
    audio_unsup = ldm_stable.decode_to_mel(x0_dec_u)

os.makedirs("results", exist_ok=True)
out_unsup = "results/unsupervised_edit.wav"
torchaudio.save(out_unsup, audio_unsup, sample_rate=sr_u)
print(f"Saved to {out_unsup}")

In [ ]:
# @markdown ### Listen to unsupervised editing result
print("Original audio:")
display(Audio(audio_orig.squeeze().cpu().numpy(), rate=sr))

print(f"Unsupervised edit (PCs {EVS}, amount={AMOUNT}):")
display(Audio(audio_unsup.squeeze().cpu().numpy(), rate=sr_u))

## 5. Music Remixing

Three remix modes are available:
- **Style Transfer**: Keep the structure of source A, apply a new style via text prompt
- **Latent Blend**: Mix two audio sources in the latent space with a controllable ratio
- **Temporal Mix**: Seamlessly transition between two styles within a single track

Supports MP3, WAV, FLAC, OGG input. Output format matches input by default.

In [ ]:
# @markdown ### Choose Remix Mode
REMIX_MODE = "style_transfer"  # @param ["style_transfer", "latent_blend", "temporal"]
print(f"Selected remix mode: {REMIX_MODE}")

In [ ]:
# @markdown ### Upload source audio(s) (WAV, MP3, FLAC, or OGG)
from google.colab import files

print("Upload Source A (primary audio):")
uploaded_a = files.upload()
REMIX_SRC_A = list(uploaded_a.keys())[0]
print(f"Source A: {REMIX_SRC_A}")

REMIX_SRC_B = None
if REMIX_MODE == "latent_blend":
    print("\nUpload Source B (blend audio):")
    uploaded_b = files.upload()
    REMIX_SRC_B = list(uploaded_b.keys())[0]
    print(f"Source B: {REMIX_SRC_B}")

# Preview
wav_a, sr_a = torchaudio.load(REMIX_SRC_A)
print("\nSource A:")
display(Audio(wav_a.squeeze().numpy(), rate=sr_a))
if REMIX_SRC_B:
    wav_b, sr_b = torchaudio.load(REMIX_SRC_B)
    print("Source B:")
    display(Audio(wav_b.squeeze().numpy(), rate=sr_b))

In [ ]:
# @markdown ### Remix Parameters
REMIX_SOURCE_PROMPT = ""  # @param {type:"string"}
REMIX_TARGET_PROMPT = "acoustic guitar version, warm tone"  # @param {type:"string"}
REMIX_CFG_SRC = 3.0  # @param {type:"number"}
REMIX_CFG_TAR = 12.0  # @param {type:"number"}
REMIX_TSTART = 100  # @param {type:"integer"}
# For latent_blend:
BLEND_RATIO = 0.5  # @param {type:"number", min:0, max:1, step:0.1}
# For temporal mode:
MIX_POINT = 0.5  # @param {type:"number", min:0.1, max:0.9, step:0.1}
# fraction of audio for source A before transitioning to B

In [ ]:
# @markdown ### Run Remix
from ddm_inversion.inversion_utils import inversion_forward_process, inversion_reverse_process

set_reproducability(seed=42, extreme=False)
os.makedirs("results", exist_ok=True)

# Determine output format from source file extension
src_ext = os.path.splitext(REMIX_SRC_A)[1].lstrip('.') or 'wav'

if REMIX_MODE == "style_transfer":
    print("[Style Transfer] Inverting source A...")
    x0_r, sr_r, dur_r = load_audio(REMIX_SRC_A, ldm_stable.get_fn_STFT(), device=device,
                                   stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())
    torch.cuda.empty_cache()
    with inference_mode():
        w0_r = ldm_stable.vae_encode(x0_r)
    with inference_mode():
        wt_r, zs_r, wts_r, extra_r = inversion_forward_process(
            ldm_stable, w0_r, etas=1.0, prompts=[REMIX_SOURCE_PROMPT],
            cfg_scales=[REMIX_CFG_SRC], prog_bar=True,
            num_inference_steps=NUM_STEPS, numerical_fix=True, duration=dur_r)
    print("Generating remixed audio...")
    tstart_r = torch.tensor([REMIX_TSTART], dtype=torch.int)
    skip_r = NUM_STEPS - tstart_r
    with inference_mode():
        w0_remix, _ = inversion_reverse_process(
            ldm_stable, xT=wts_r, tstart=tstart_r, fix_alpha=0.1, etas=1.0,
            prompts=[REMIX_TARGET_PROMPT], neg_prompts=[""],
            cfg_scales=[REMIX_CFG_TAR], prog_bar=True,
            zs=zs_r[:int(NUM_STEPS - min(skip_r))], duration=dur_r, extra_info=extra_r)

elif REMIX_MODE == "latent_blend":
    print(f"[Latent Blend] Blending with ratio {BLEND_RATIO:.1f}...")
    x0_a, sr_r, dur_a = load_audio(REMIX_SRC_A, ldm_stable.get_fn_STFT(), device=device,
                                   stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())
    x0_b, _, dur_b = load_audio(REMIX_SRC_B, ldm_stable.get_fn_STFT(), device=device,
                                 stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())
    dur_r = min(dur_a, dur_b)
    torch.cuda.empty_cache()
    with inference_mode():
        w0_a = ldm_stable.vae_encode(x0_a)
        w0_b = ldm_stable.vae_encode(x0_b)
    min_len = min(w0_a.shape[-1], w0_b.shape[-1])
    w0_a = w0_a[:, :, :, :min_len]
    w0_b = w0_b[:, :, :, :min_len]
    blend = (1 - BLEND_RATIO) * w0_a + BLEND_RATIO * w0_b
    print("Inverting blended latent...")
    with inference_mode():
        wt_r, zs_r, wts_r, extra_r = inversion_forward_process(
            ldm_stable, blend, etas=1.0, prompts=[REMIX_SOURCE_PROMPT],
            cfg_scales=[REMIX_CFG_SRC], prog_bar=True,
            num_inference_steps=NUM_STEPS, numerical_fix=True, duration=dur_r)
    print("Reconstructing...")
    tstart_r = torch.tensor([0], dtype=torch.int)
    with inference_mode():
        w0_remix, _ = inversion_reverse_process(
            ldm_stable, xT=wts_r, tstart=tstart_r, fix_alpha=0.0, etas=1.0,
            prompts=[REMIX_SOURCE_PROMPT], neg_prompts=[""],
            cfg_scales=[REMIX_CFG_SRC], prog_bar=True,
            zs=zs_r, duration=dur_r, extra_info=extra_r)

elif REMIX_MODE == "temporal":
    print(f"[Temporal Mix] Transition at {MIX_POINT:.0%}...")
    x0_r, sr_r, dur_r = load_audio(REMIX_SRC_A, ldm_stable.get_fn_STFT(), device=device,
                                   stft=('stable-audio' not in MODEL_ID), model_sr=ldm_stable.get_sr())
    torch.cuda.empty_cache()
    with inference_mode():
        w0_r = ldm_stable.vae_encode(x0_r)
    prompts_r = [REMIX_SOURCE_PROMPT, REMIX_TARGET_PROMPT]
    cfg_scales_r = [REMIX_CFG_SRC, REMIX_CFG_TAR]
    print("Inverting source A...")
    with inference_mode():
        wt_r, zs_r, wts_r, extra_r = inversion_forward_process(
            ldm_stable, w0_r, etas=1.0, prompts=prompts_r,
            cfg_scales=cfg_scales_r, prog_bar=True,
            num_inference_steps=NUM_STEPS, numerical_fix=True,
            cutoff_points=[MIX_POINT], duration=dur_r)
    print("Generating temporally remixed audio...")
    tstart_r = torch.tensor([REMIX_TSTART], dtype=torch.int)
    skip_r = NUM_STEPS - tstart_r
    with inference_mode():
        w0_remix, _ = inversion_reverse_process(
            ldm_stable, xT=wts_r, tstart=tstart_r, fix_alpha=0.1, etas=1.0,
            prompts=prompts_r, neg_prompts=["", ""],
            cfg_scales=cfg_scales_r, prog_bar=True,
            zs=zs_r[:int(NUM_STEPS - min(skip_r))],
            cutoff_points=[MIX_POINT], duration=dur_r, extra_info=extra_r)

# Decode
print("Decoding...")
with inference_mode():
    x0_dec_r = ldm_stable.vae_decode(w0_remix)
    if x0_dec_r.dim() < 4:
        x0_dec_r = x0_dec_r[None, :, :, :]
    with torch.no_grad():
        audio_remix = ldm_stable.decode_to_mel(x0_dec_r)

# Save with matching format (MP3 input -> MP3 output)
out_remix = f"results/remix_{REMIX_MODE}.{src_ext}"
torchaudio.save(out_remix, audio_remix, sample_rate=sr_r, format=src_ext if src_ext != 'wav' else None)
print(f"Saved to {out_remix}")

In [ ]:
# @markdown ### Listen to remix result
print("Source A:")
display(Audio(wav_a.squeeze().numpy(), rate=sr_a))
if REMIX_SRC_B:
    print("Source B:")
    display(Audio(wav_b.squeeze().numpy(), rate=sr_b))
print(f"Remix ({REMIX_MODE}, prompt: \"{REMIX_TARGET_PROMPT}\"):")
display(Audio(audio_remix.squeeze().cpu().numpy(), rate=sr_r))

## 6. Download Results

Run the cell below to download all generated audio files.

In [ ]:
from google.colab import files

print("Downloading results...")
for f in sorted(os.listdir("results")):
    if f.endswith((".wav", ".mp3", ".flac", ".ogg", ".png")):
        print(f"  - {f}")
        files.download(f"results/{f}")